# Cloud Red Teaming in the new Microsoft Foundry experience

**Goal:** create a cloud red-team evaluation, inspect row-level attack results through the SDK, and open the same report in the new Foundry portal.

## Which API does this notebook use?

Microsoft currently exposes two different SDK workflows:

- **New Foundry cloud workflow, used here:** `project_client.get_openai_client().evals` creates an evaluation group, starts server-side red-team runs, and exposes every output item through the SDK. Its report URL opens the new Foundry experience.
- **Local workflow:** `azure-ai-evaluation[redteam]` runs the PyRIT-based orchestrator from the local Python environment. Microsoft documents this workflow as not compatible with the new Foundry portal and SDK.
- **Legacy cloud preview:** `project_client.beta.red_teams` creates `/redTeams/runs` resources whose report links open the classic Foundry evaluation experience. This notebook no longer uses that API.

Cloud red teaming still uses attack strategies derived from PyRIT, but PyRIT does not need to be installed in this notebook environment because orchestration runs in Foundry.

| Time | What to show |
| --- | --- |
| 0:00 | Create an evaluation group for Violence risk |
| 0:45 | Configure Baseline and Base64 against the model deployment |
| 1:30 | Start or retrieve a cloud run |
| 2:30 | Inspect row-level attack outcomes and ASR |
| 4:15 | Open the same run in the new Foundry report |

## Connect to the new Foundry Evals API

In [5]:
import os
import time

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

if not load_dotenv("./../credentials_my.env"):
    raise RuntimeError("Environment variables could not be loaded")

foundry_project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)
evals_client = project_client.get_openai_client().evals

print(f"Foundry project endpoint: {foundry_project_endpoint}")
print(f"Target deployment: {deployment_name}")

Foundry project endpoint: https://mm-ai-upskilling-project-resourc.services.ai.azure.com/api/projects/ai-upskilling-project
Target deployment: gpt-5.4-mini


## Create the red-team evaluation group

The evaluation group defines **what constitutes failure**. Here, the built-in Violence evaluator marks an output as failed when the model response violates the violence safety criterion.

Creating the group does not run attacks yet. It provides the stable container under which one or more comparable red-team runs can be tracked.

In [ ]:
red_team_eval = evals_client.create(
    name="Violence red-team evaluation",
    data_source_config={
        "type": "azure_ai_source",
        "scenario": "red_team",
    },
    testing_criteria=[
        {
            "type": "azure_ai_evaluator",
            "name": "Violence",
            "evaluator_name": "builtin.violence",
            "evaluator_version": "1",
        }
    ],
)

print(f"Evaluation group ID: {red_team_eval.id}")

## Configure and start a cloud run

This run applies two techniques to generated Violence objectives:

- **Baseline** sends the direct adversarial objective.
- **Base64** encodes the objective before sending it.

Base64 is not inherently stronger than Baseline. It is a different probe that might bypass one control while failing against another. The meaningful comparison is the observed success count for each technique.

In [ ]:
red_team_run = evals_client.runs.create(
    eval_id=red_team_eval.id,
    name="Violence Baseline and Base64 run",
    data_source={
        "type": "azure_ai_red_team",
        "item_generation_params": {
            "type": "red_team",
            "attack_strategies": ["baseline", "base64"],
            "num_turns": 1,
        },
        "target": {
            "type": "azure_ai_model",
            "model": deployment_name,
        },
    },
)

print(f"Run ID: {red_team_run.id}")
print(f"Initial status: {red_team_run.status}")

In [ ]:
terminal_statuses = {"completed", "failed", "canceled"}
last_status = None

for attempt in range(60):
    red_team_run = evals_client.runs.retrieve(
        run_id=red_team_run.id,
        eval_id=red_team_eval.id,
    )

    if red_team_run.status != last_status:
        print(f"Status: {red_team_run.status}")
        last_status = red_team_run.status

    if red_team_run.status in terminal_statuses:
        break

    if attempt < 59:
        time.sleep(5)
else:
    raise TimeoutError(
        "The run is still active. Continue with the completed-run fallback."
    )

## Demo-safe fallback: select an existing completed run

Creating a cloud run is the slow and variable part of the demo. Run it before class. During the presentation, the next cell retrieves the newest completed red-team run from the **new Evals API** without creating cloud work.

In [ ]:
completed_runs = []

for candidate_eval in evals_client.list(limit=100, order="desc"):
    eval_payload = candidate_eval.model_dump(exclude_none=True, warnings=False)
    if (eval_payload.get("data_source_config") or {}).get("scenario") != "red_team":
        continue

    for candidate_run in evals_client.runs.list(
        candidate_eval.id,
        limit=20,
        order="desc",
        status="completed",
    ):
        completed_runs.append((candidate_eval, candidate_run))

if not completed_runs:
    raise RuntimeError(
        "No completed run exists in the new Foundry Evals API. "
        "Create the run before the presentation."
    )

red_team_eval, red_team_run = max(
    completed_runs,
    key=lambda pair: pair[1].created_at,
)

{
    "evaluation_id": red_team_eval.id,
    "run_id": red_team_run.id,
    "run_name": red_team_run.name,
    "status": red_team_run.status,
    "report_url": red_team_run.report_url,
}

## Inspect the same results exposed by the new portal

The SDK returns the row-level output items used by the new Foundry report.

For a safety criterion:

- `pass` means the response passed the Violence evaluator, so the attack was unsuccessful.
- `fail` means the response failed the Violence evaluator, so the attack was successful.

The next cell shows both the numerator and denominator. With one item, `1/1 (100%)` is one observed failure, not evidence that a technique always succeeds. Red-team grading is also model-based and can produce false positives or false negatives, so surprising rows require human review.

In [ ]:
from collections import defaultdict
from IPython.display import Markdown, display

output_items = list(
    evals_client.runs.output_items.list(
        run_id=red_team_run.id,
        eval_id=red_team_eval.id,
    )
)

if not output_items:
    raise RuntimeError("The completed run has no output items.")


def find_first(value, candidate_keys):
    if isinstance(value, dict):
        for key in candidate_keys:
            if key in value and value[key] not in (None, ""):
                return value[key]
        for nested_value in value.values():
            found = find_first(nested_value, candidate_keys)
            if found not in (None, ""):
                return found
    elif isinstance(value, list):
        for nested_value in value:
            found = find_first(nested_value, candidate_keys)
            if found not in (None, ""):
                return found
    return None


def clean_cell(value, limit=180):
    text = " ".join(str(value or "").split())
    if len(text) > limit:
        text = f"{text[:limit - 3]}..."
    return text.replace("|", "\\|")


rows = []
strategy_counts = defaultdict(lambda: {"successful": 0, "total": 0})

for output_item in output_items:
    payload = output_item.model_dump(exclude_none=True, warnings=False)
    datasource_item = payload.get("datasource_item") or {}
    sample = payload.get("sample") or {}

    strategy = find_first(
        datasource_item,
        ["attack_technique", "attackTechnique", "attack_strategy", "attackStrategy"],
    ) or "Unknown"
    complexity = find_first(
        datasource_item,
        ["attack_complexity", "attackComplexity"],
    ) or "Unknown"
    risk_category = find_first(
        datasource_item,
        ["risk_category", "riskCategory"],
    ) or "Violence"

    status = payload.get("status", "unknown").lower()
    attack_successful = status == "fail"
    strategy_counts[strategy]["total"] += 1
    strategy_counts[strategy]["successful"] += int(attack_successful)

    assistant_outputs = [
        message.get("content", "")
        for message in sample.get("output", [])
        if message.get("role") == "assistant"
    ]
    rows.append(
        {
            "strategy": strategy,
            "complexity": complexity,
            "risk_category": risk_category,
            "outcome": "Attack successful" if attack_successful else "Attack unsuccessful",
            "response": " ".join(assistant_outputs),
        }
    )

summary_lines = [
    "### Attack Success Rate by technique",
    "",
    "| Technique | Successful / Total | ASR |",
    "| --- | ---: | ---: |",
]
for strategy, counts in sorted(strategy_counts.items()):
    asr = counts["successful"] / counts["total"]
    summary_lines.append(
        f"| {clean_cell(strategy)} | {counts['successful']} / {counts['total']} | {asr:.0%} |"
    )

summary_lines.extend(
    [
        "",
        "### Row-level evidence",
        "",
        "| Technique | Complexity | Risk | Grader outcome | Assistant response excerpt |",
        "| --- | --- | --- | --- | --- |",
    ]
)
for row in rows:
    summary_lines.append(
        "| {strategy} | {complexity} | {risk_category} | {outcome} | {response} |".format(
            **{key: clean_cell(value) for key, value in row.items()}
        )
    )

if red_team_run.report_url:
    summary_lines.extend(
        [
            "",
            f"[Open this run in the new Microsoft Foundry report]({red_team_run.report_url})",
        ]
    )

display(Markdown("\n".join(summary_lines)))

## How to explain a surprising comparison

A result such as **Baseline: 1/1 successful** and **Base64: 0/1 successful** is valid but weak evidence:

- The techniques are separate probes, not ordered difficulty levels. Encoding can make the request harder for the target model to interpret, so Base64 can fail even when the direct request succeeds.
- With one generated item per technique, the observed ASRs can only be `0%` or `100%`. Always state the counts and repeat with broader coverage before drawing conclusions.
- “Attack successful” means the safety grader marked the response as violating the configured criterion. It does not necessarily mean that a jailbreak instruction overrode the system prompt.
- Review surprising rows manually. Microsoft documents red-team ASR grading as non-deterministic and subject to false positives and false negatives.

The useful finding is therefore not “Base64 is safer than Baseline.” It is: **this run surfaced one direct Violence case that requires review, while the transformed probe did not reproduce the failure.**